In [2]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

# DEM collection, ~30m resolution
dataset = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(panama_geom)
)

elevation = dataset.select('DEM')

elev_projection = elevation.first().projection()

# ~28m resolution
hansen = ee.Image("UMD/hansen/global_forest_change_2025_v1_13").clip(panama_geom)

lossyear = hansen.select("lossyear")

recent_loss = lossyear.gte(20).selfMask()

distance_loss = (
    recent_loss.fastDistanceTransform(256)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())
    .rename("dist_loss")
)

# export this as an image asset with the elev_projection and scale.
task = ee.batch.Export.image.toAsset(
    image = distance_loss,
    description = 'distance_loss',
    assetId = "projects/deforestation-panama/assets/distance_loss",
    region = panama_geom,
    scale = elev_projection.nominalScale(),
    crs = elev_projection.crs(),
    crsTransform = elev_projection.getInfo()['transform'],
    maxPixels=1e12
)

task.start()

# reprojecting to fit other data layers
# sample_image = elevation
# collection_projection = sample_image.projection()

# reprojected_distance_loss = (
#     distance_loss.resample("bilinear")
#     .reproject(collection_projection)
#     .rename("distance_loss_reprojected")
#     .clip(panama_geom)
# )

# Map.addLayer(recent_loss, {"palette": ["red"]}, "Recent Loss")
# Map.addLayer(distance_loss, {"min": 0, "max": 5000}, "Dist to Loss")
# Map.addLayer(reprojected_distance_loss, {"min": 0, "max": 5000}, "Dist to Loss Reproj")

# Map

In [4]:
Map = geemap.Map()
Map.centerObject(panama_geom, 7)

export = ee.Image(projects/deforestation-panama/assets/distance_loss)

Map.addLayer(export, {}, "Dist Deforested Areas")

Map

NameError: name 'projects' is not defined